In [13]:
print("hello world")

hello world


In [26]:
! uv pip install rapidocr-onnxruntime

Using Python 3.12.11 environment at: D:\Generative_AI\Projects\LLMops_multi_doc\.venv
Resolved 19 packages in 1.30s
Uninstalled 2 packages in 75ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 623ms
 - numpy==2.3.5
 ~ numpy==2.2.6


In [27]:
import os

from dotenv import load_dotenv

load_dotenv()

True

In [28]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

### DATA INGESTION

In [29]:
from langchain.document_loaders import TextLoader

In [30]:
loader = TextLoader(r"D:\Generative_AI\Projects\LLMops_multi_doc\data\agentic_ai.txt", encoding="utf8")
loader

In [31]:
document = loader.load()

In [32]:
document[0].page_content[:5000] # Print the first 500 characters of the first document

'# Content for the Agentic AI text file\nfile_content = """\n# Agentic AI: A Technical Overview\n\n## 1. What is Agentic AI?\nUnlike traditional LLM interactions (Zero-shot/Few-shot) where the model acts as a passive engine awaiting a prompt, Agentic AI refers to systems where the LLM acts as a reasoning engine that can:\n1. Perceive its environment.\n2. Reason about how to solve a problem.\n3. Break down complex goals into smaller tasks (Planning).\n4. Use external tools (Function Calling) to execute actions.\n5. Reflect on the results and iterate.\n\n## 2. Core Components of an Agent\n\n### A. Profile/Persona\nThe specific role and constraints assigned to the agent (e.g., "You are a Senior Python Engineer...").\n\n### B. Memory\n- **Short-term:** Context window, passing messages between steps.\n- **Long-term:** Vector databases (RAG) or SQL databases to recall past interactions or knowledge.\n\n### C. Planning\n- **Chain of Thought (CoT):** Breaking down steps linearly.\n- **Tree of 

In [33]:
document

[Document(metadata={'source': 'D:\\Generative_AI\\Projects\\LLMops_multi_doc\\data\\agentic_ai.txt'}, page_content='# Content for the Agentic AI text file\nfile_content = """\n# Agentic AI: A Technical Overview\n\n## 1. What is Agentic AI?\nUnlike traditional LLM interactions (Zero-shot/Few-shot) where the model acts as a passive engine awaiting a prompt, Agentic AI refers to systems where the LLM acts as a reasoning engine that can:\n1. Perceive its environment.\n2. Reason about how to solve a problem.\n3. Break down complex goals into smaller tasks (Planning).\n4. Use external tools (Function Calling) to execute actions.\n5. Reflect on the results and iterate.\n\n## 2. Core Components of an Agent\n\n### A. Profile/Persona\nThe specific role and constraints assigned to the agent (e.g., "You are a Senior Python Engineer...").\n\n### B. Memory\n- **Short-term:** Context window, passing messages between steps.\n- **Long-term:** Vector databases (RAG) or SQL databases to recall past inter

In [34]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [35]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 200, chunk_overlap = 20)

In [36]:
text_chunks = text_splitter.split_documents(documents=document)
text_chunks

[Document(metadata={'source': 'D:\\Generative_AI\\Projects\\LLMops_multi_doc\\data\\agentic_ai.txt'}, page_content='# Content for the Agentic AI text file\nfile_content = """\n# Agentic AI: A Technical Overview'),
 Document(metadata={'source': 'D:\\Generative_AI\\Projects\\LLMops_multi_doc\\data\\agentic_ai.txt'}, page_content='## 1. What is Agentic AI?'),
 Document(metadata={'source': 'D:\\Generative_AI\\Projects\\LLMops_multi_doc\\data\\agentic_ai.txt'}, page_content='Unlike traditional LLM interactions (Zero-shot/Few-shot) where the model acts as a passive engine awaiting a prompt, Agentic AI refers to systems where the LLM acts as a reasoning engine that can:'),
 Document(metadata={'source': 'D:\\Generative_AI\\Projects\\LLMops_multi_doc\\data\\agentic_ai.txt'}, page_content='1. Perceive its environment.\n2. Reason about how to solve a problem.\n3. Break down complex goals into smaller tasks (Planning).\n4. Use external tools (Function Calling) to execute actions.'),
 Document(meta

In [38]:
len(text_chunks[0].page_content)

92

In [41]:
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS

In [42]:
embeddings = OpenAIEmbeddings()

In [43]:
vectorstore = FAISS.from_documents(text_chunks, embedding=embeddings)

In [44]:
retriever = vectorstore.as_retriever()

In [45]:
# peform similarity search
query = "What is the key characteristics of Agentic AI"
docs = vectorstore.similarity_search(query,k=3)

In [46]:
for idx,doc in enumerate(docs):
    print(f"Document {idx+1} : ")
    print(doc.page_content)
    print("_"* 50)

Document 1 : 
## 1. What is Agentic AI?
__________________________________________________
Document 2 : 
Unlike traditional LLM interactions (Zero-shot/Few-shot) where the model acts as a passive engine awaiting a prompt, Agentic AI refers to systems where the LLM acts as a reasoning engine that can:
__________________________________________________
Document 3 : 
# Content for the Agentic AI text file
file_content = """
# Agentic AI: A Technical Overview
__________________________________________________


In [47]:
from langchain.prompts import ChatPromptTemplate

In [48]:
template = """You are an assistant for question-answering tasks.
    use the following pieces of retrieved context to answer the question.
    if you don't know the answer, just say that you don't know.
    Use ten sentences maximum and keep the answer concise.
    Question : {question}
    Context : {context}
    Answer :
    """

In [49]:
prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\n    use the following pieces of retrieved context to answer the question.\n    if you don't know the answer, just say that you don't know.\n    Use ten sentences maximum and keep the answer concise.\n    Question : {question}\n    Context : {context}\n    Answer :\n    "), additional_kwargs={})])

In [50]:
from langchain.schema.output_parser import StrOutputParser

In [52]:
output_parser = StrOutputParser()

In [53]:
from langchain_openai import ChatOpenAI

llm_model = ChatOpenAI(model="gpt-4o-mini")

In [54]:
from langchain.schema.runnable import RunnablePassthrough

In [ ]:
rag_chain = (
    {"context" : retriever, "question" : RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
    )

In [56]:
rag_chain.invoke("tell me about Agentic AI")

'Agentic AI refers to advanced systems where large language models (LLMs) operate as reasoning engines rather than merely responding to prompts in a passive manner. This contrasts with traditional LLM interactions, which typically involve zero-shot or few-shot learning approaches. In Agentic AI, the model can actively engage in reasoning and decision-making processes. Additionally, it may involve multi-agent systems (MAS), where specialized agents collaborate to achieve common goals. These systems enhance the capabilities of AI by enabling more dynamic and autonomous interactions. Overall, Agentic AI represents a significant evolution in how AI can be utilized for complex tasks.'